In [1]:
import scipy.spatial
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import time
time1 = time.time_ns()

np.random.seed(445468)

pts = np.random.rand(20,2)
Vor = scipy.spatial.Voronoi(pts) #Delaunay triangularisation already implemented



events_this_frame = []



def keeps(Vor, r=24):
    '''returns indices of unbroken nuclei; ones producing cells with sufficient
    roundness, and with bounded edges'''
    kept_points = []  # indices of points that arent broken
    for i in range(len(Vor.points)): #for each cell
        region_idx = Vor.point_region[i]
        vert_idx = Vor.regions[region_idx]
        if -1 in vert_idx or len(vert_idx) < 3:
            continue
        are = area(Vor, i)
        perim = perimeter(Vor, i) #the perimeter must be squared so that the ratio is scale invariant
        if (perim)**2/are <= r: #area/perimeter ratio ensures sufficient roundness
            kept_points.append(i)
    return kept_points






def make_nice_tesseltion(Vor, kept, ax=None):
    '''plots kept cells'''

    if ax is None:
        fig, ax = plt.subplots()

    #pts = Vor.points[kept]
    #ax.scatter(pts[:,0], pts[:,1], color='blue', marker='o') #nucleus

    for key, value in Vor.ridge_dict.items():
        if any(n in key for n in kept):
            coords = Vor.vertices[value]
            ax.plot(coords[:,0], coords[:,1], color='blue')
    return ax






def show(Vor, axis, n, i = None, colour = 'red'):
    '''shows cell of nth nucleus. Optionally show specified vertex'''

    #axis.scatter(Vor.points[n][0],Vor.points[n][1], color='red', marker='o')
    edges = {key:value for key, value in Vor.ridge_dict.items() if n in key}
    for key in edges:
        edge = Vor.vertices[edges[key]]
        axis.plot(edge[:, 0], edge[:, 1], color=colour)

    if i is not None:
        coord = Vor.vertices[i]
        axis.scatter(coord[0],coord[1], color='green', marker='o', zorder=2)

    return axis





area_thresh = 0.00001
small = [] #small cells
def area(Vor, n, i = None):
    '''returns area of cell, optionally along with perimeter gradient of specified vertex'''
    region = Vor.regions[Vor.point_region[n]]

    #Shoelace formula (signed area)
    x = Vor.vertices[region,0]
    y = Vor.vertices[region,1]
    area = 0.5 * (np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

    if np.abs(area) < area_thresh: small.append(n)

    if i is not None:
        if area < 0: region = region[::-1] #area is negative if cyclic order is reversed

        # find position of vertex i in the ordered list
        k = region.index(i)

        # cyclic neighbors
        kplus1  = region[(k+1) % len(region)]
        kminus1 = region[(k-1) % len(region)]

        # analytic derivatives
        dA_dxi = 0.5 * (Vor.vertices[kplus1,1] - Vor.vertices[kminus1,1])
        dA_dyi = 0.5 * (Vor.vertices[kminus1,0] - Vor.vertices[kplus1,0])

        return np.abs(area), [dA_dxi, dA_dyi]

    else: return np.abs(area)






len_thresh = 0.005 #length at which edge becomes degenerate
short = [] #short edges
def perimeter(Vor, n, i = None):
    '''returns perimeter of cell, optionally along with perimeter gradient of specified vertex'''

    dP_dxi=0
    dP_dyi=0

    perim=0

    for key, value in Vor.ridge_dict.items():
        if n in key:

            edge = Vor.vertices[value] #edge between each neighboring cell
            length = np.sqrt((edge[0,0]-edge[1, 0])**2 + (edge[0,1]-edge[1, 1])**2) #Euclidian distance between vertices/line length formula
            perim+=length #add to perimeter

            if (np.abs(length) < len_thresh) and (key not in short):
                short.append(key)
                #print(np.abs(length))

            if i in value: #perim grad at i only depends on its 2 connected vertices
                j = value[1-value.index(i)]


                dP_dxi += (Vor.vertices[i,0]-Vor.vertices[j,0])/length
                dP_dyi += (Vor.vertices[i,1]-Vor.vertices[j,1])/length



    if i is not None: return perim, [dP_dxi, dP_dyi]
    else: return perim





a = 0.086

oblong = [] #oblong cells to split
def energy(Vor, pts, i = None):
    '''calculates energy based off of kept points.
    Alternatively returns energy gradient of specified vertex i'''
    potential = 0
    ka = 1 #Modulus of Elasticity for Area (osmotic preassure)
    kp = 1 #Modulus of Elasticity for Perimeter (cell wall tension)
    P0 = 1
    A0 = (P0/S0)**2

    if i is None:
        for pt in pts:
            potential += 0.5*ka*((area(Vor, pt)-A0)**2) + 0.5*kp*((perimeter(Vor, pt)-P0)**2)
        return potential

    elif i is not None:

        dE_dxi=0
        dE_dyi=0
        for n in pts:
            if i in Vor.regions[Vor.point_region[n]]:

                A, dA_dri = area(Vor, n, i)
                P, dP_dri = perimeter(Vor, n, i)

                dE_dxi += ka*(A-A0)*dA_dri[0] + kp*(P-P0)*dP_dri[0]
                dE_dyi += ka*(A-A0)*dA_dri[1] + kp*(P-P0)*dP_dri[1]

                if (A >= a) and (n not in oblong): oblong.append(n)

        return dE_dxi, dE_dyi







def finite_diff(Vor, pts, i):
    '''finite difference method'''
    potential = 0
    ka = 1 #Modulus of Elasticity for Area (osmotic preassure)
    kp = 1 #Modulus of Elasticity for Perimeter (cell wall tension)
    P0 = 0.4
    A0 = (P0/S0)**2

    if i is None:
        for pt in pts:
            A, P = area(Vor, pt), perimeter(Vor, pt)
            potential += 0.5*ka*((area(Vor, pt)-A0)**2) + 0.5*kp*((perimeter(Vor, pt)-P0)**2)
            if (A >= a) and (pt not in oblong): oblong.append(pt)
        return potential

    delta = 1e-6

    neighbours = [neigh for neigh in pts if i in Vor.regions[Vor.point_region[neigh]]]
    #only 3 cells affected by movement of 1 vertex

    E1 = energy(Vor, neighbours)

    Vor.vertices[i,0] += delta #small perturbation in x direction
    E2 = energy(Vor, neighbours) #energy after perturbation E(xi + delta)
    dE = E2-E1
    grad_x = dE/delta #energy gradient in x direction

    Vor.vertices[i,0] -= delta #reset vertex to original position

    Vor.vertices[i,1] += delta #small perturbation in y direction
    E2 = energy(Vor, neighbours) #energy after perturbation E(xi + delta)
    dE = E2-E1
    grad_y = dE/delta #energy gradient in y direction

    Vor.vertices[i,1] -= delta #reset vertex to original position

    return grad_x, grad_y









def find_relevant_vertices(Vor, pts):
    '''returns all vertices relevant to kept points'''
    vtx =[]
    for k in pts:
        r = Vor.point_region[k]
        for i in Vor.regions[r]:
                if i not in vtx: vtx += [i]
    return vtx



def ray_cast(Vor, i, n):
    'checks if vertex i is located within cell n, using ray casting method'
    [x,y] = Vor.vertices(i)

    region = Vor.regions[Vor.point_region[n]]
    k = len(region) #degree of the polygon
    p1x, p1y = Vor.vertices[region[0]]

    inside = False
    for i in range(k + 1):
        p2x, p2y = Vor.vertices[region[i % k]]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
                    if p1x == p2x or x <= xinters:
                        inside = not inside
        p1x, p1y = p2x, p2y
    return inside







def Adjacents(Vor, pair):

    (n1, n2) = (pair[0], pair[1])
    (v1, v2) = Vor.ridge_dict[(n1, n2)]

    adj = []

    for key, value in Vor.ridge_dict.items():

        if ((v1 in value) or (v2 in value)):

            adj += [n for n in key if n not in adj]

    adj.remove(n1)
    adj.remove(n2)

    if len(adj)>2: adj = pair

    return tuple(adj)




def reorder(Vor):
    "reorders Vor.regions cyclically"
    #reorder regions cyclically if any were edited
    for n in kept:
        ordered = []
        # all edges belonging to cell n
        edges = [value[:] for key, value in Vor.ridge_dict.items() if n in key]

        # start with any edge
        this, nxt = edges.pop(0)
        ordered.append(this)

        while edges:
            for pair in edges:
                if nxt in pair:
                    ordered.append(nxt)
                    # compute the next vertex
                    new = pair[1 - pair.index(nxt)]
                    # remove the used edge
                    edges.remove(pair)
                    nxt = new
                    break
        Vor.regions[Vor.point_region[n]] = ordered






def T1(Vor, pair, adj):

    (n1, n2) = pair
    (a1, a2) = adj

    #Remove old ridge and get its vertices
    if pair not in Vor.ridge_dict: pair = (pair[1], pair[0]) #check orientation of key
    v1, v2 = Vor.ridge_dict.pop(pair)

    Vor.ridge_dict[adj] = [v1, v2] #Add new ridge (a1, a2) with same vertices


    if v1 not in Vor.regions[Vor.point_region[a1]]: #if incorrect order
        temp = v1
        v1 = v2
        v2 = temp

    # a1 and a2 each gain one
    Vor.regions[Vor.point_region[a1]].append(v2)
    Vor.regions[Vor.point_region[a2]].append(v1)
    # n2 loses the other vertex
    Vor.regions[Vor.point_region[n2]].remove(v1)
    Vor.regions[Vor.point_region[n1]].remove(v2)

    if (a1, n2) in Vor.ridge_dict:
        old = Vor.ridge_dict.pop((a1, n2))
        old.remove(v1)
        old.append(v2)
        Vor.ridge_dict[(a1, n2)] = old

    elif (n2, a1) in Vor.ridge_dict: #key potentially reversed for ridge dict
        old = Vor.ridge_dict.pop((n2, a1))
        old.remove(v1)
        old.append(v2)
        Vor.ridge_dict[(n2, a1)] = old

    #now for other adjacent cell
    if (a2, n1) in Vor.ridge_dict:
        old = Vor.ridge_dict.pop((a2, n1))
        old.remove(v2)
        old.append(v1)
        Vor.ridge_dict[(a2, n1)] = old

    elif (n1, a2) in Vor.ridge_dict: #key potentially reversed for ridge dict
        old = Vor.ridge_dict.pop((n1, a2))
        old.remove(v2)
        old.append(v1)
        Vor.ridge_dict[(n1, a2)] = old

    #nudge towards centroid
    verts = Vor.vertices[Vor.regions[Vor.point_region[n2]]]
    centroid = np.mean(verts, axis=0)          # coordinates
    d = centroid - Vor.vertices[v2]            # vector in R^2
    d = d / np.linalg.norm(d)
    Vor.vertices[v1] -= 0.5*len_thresh * d
    Vor.vertices[v2] += 0.5*len_thresh * d

    #vertices need to be nudged apart so that T1 doesnt automatically re-trigger

    # Record event marker at midpoint of the flipped edge
    midpoint = 0.5 * (Vor.vertices[v1] + Vor.vertices[v2])
    events_this_frame.append(("T1", midpoint))






def T2(Vor, n):
    """
    T2 transition: remove cell n, collapse its polygon into a single vertex.
    """

    #remove n from kept
    if n in kept:
        kept.remove(n)

    #vertices to collapse into one
    delete = Vor.regions[Vor.point_region[n]][:]

    #centroid of the polygon
    centroid = np.mean(Vor.vertices[delete], axis=0)

    #append new vertex
    Vor.vertices = np.vstack([Vor.vertices, centroid])
    i = len(Vor.vertices) - 1
    vtx.append(i)

    #replace deleted vertices in all remaining kept cells
    for k in kept:
        region = Vor.regions[Vor.point_region[k]]
        for v in delete:
            if v in region:
                region.remove(v)
                if i not in region:
                    region.append(i)

    for d in delete:
        if d in vtx: vtx.remove(d)

    #update ridge_dict
    new_ridge_dict = {}

    for key, val in Vor.ridge_dict.items():

        #remove ridges belonging to the deleted cell
        if n in key:
            continue

        # replace deleted vertices with new vertex i
        new_val = []
        for v in val:
            if v in delete:
                new_val.append(i)
            else:
                new_val.append(v)

        new_ridge_dict[key] = new_val

    # mutate the existing dict
    Vor.ridge_dict.clear()
    Vor.ridge_dict.update(new_ridge_dict)

    # Mark the centroid where the cell collapsed
    events_this_frame.append(("T2", centroid))






def split(Vor, n):
    """
    Split cell n along its shortest PCA axis.
    """
    region = Vor.regions[Vor.point_region[n]]
    verts = Vor.vertices[region]
    center = np.mean(verts, axis=0)

    #compute shortest PCA axis
    _, _, vh = np.linalg.svd(verts - center)
    axis = vh[1]  # shortest axis (minor principal component)

    #find intersections with polygon boundary
    intersections = []
    keys = []
    for j in range(len(region)):
        i1, i2 = region[j], region[(j+1) % len(region)]
        v1 = Vor.vertices[i1]
        v2 = Vor.vertices[i2]

        p = center
        d = axis
        e = v2 - v1

        A = np.array([d, -e]).T
        b = v1 - p

        if np.linalg.matrix_rank(A) < 2:
            continue

        t, s = np.linalg.solve(A, b)

        if 0 <= s <= 1:
            intersections.append(p + t*d)

            for key, val in Vor.ridge_dict.items():
                if (i1 in val) and (i2 in val):
                    keys.append(key)

    if len(intersections) != 2: return
    if len(keys) != 2: return

    vA, vB = intersections

    #insert new vertices
    iA = len(Vor.vertices)
    iB = iA + 1
    Vor.vertices = np.vstack([Vor.vertices, vA, vB])

    #region splitting based on signed height

    #normal vector perpendicular to PCA axis
    normal = np.array([-axis[1], axis[0]])

    def signed_height(v):
        return np.dot(v - center, normal)

    reg1 = []
    reg2 = []

    for vid in region:
        v = Vor.vertices[vid]
        if signed_height(v) >= 0:
            reg1.append(vid)
        else:
            reg2.append(vid)

    #insert new vertices into both daughters
    reg1.insert(1, iA)
    reg1.insert(2, iB)

    reg2.insert(1, iB)
    reg2.insert(2, iA)

    #create new region index
    new_region_index = len(Vor.regions)
    Vor.regions.append(reg2)
    #assign new region to a new point_region entry
    Vor.point_region = np.append(Vor.point_region, new_region_index)
    new_cell = len(Vor.point_region)-1

    #replace original region with reg1
    Vor.regions[Vor.point_region[n]] = reg1


    #update the two ridges that were explicitly split
    for (c1, c2), new_vertex in zip(keys, [iA, iB]):

        #identify the adjacent cell
        adj = c1 if c1 != n else c2

        # Add the new vertex to the neighbor region
        Vor.regions[Vor.point_region[adj]].append(new_vertex)

        # Remove the old ridge
        old_vertices = Vor.ridge_dict.pop((c1, c2))

        # Orient old ridge so old_vertices[0] belongs to reg1 (the original cell)
        if old_vertices[0] not in reg1:
            old_vertices = old_vertices[::-1]

        # Ridge for reg1 (original cell n)
        Vor.ridge_dict[(adj, n)] = [old_vertices[0], new_vertex]

        # Ridge for reg2 (new daughter)
        Vor.ridge_dict[(adj, new_cell)] = [old_vertices[1], new_vertex]

    temp = []
    for key, value in Vor.ridge_dict.items(): #assign remaining ridges correctly
        if n in key:
            if (value[0] in reg2) and (value[1] in reg2):
                temp.append(key)
    for key in temp:
        n_key = list(key)
        n_key.remove(n)
        n_key.append(new_cell)

        val = Vor.ridge_dict.pop(key)
        Vor.ridge_dict[tuple(n_key)] =  val

    #add new ridge between daughters
    Vor.ridge_dict[(n, new_cell)] = [iA, iB]

    #update relevant cell/vertex references
    vtx.append(iA)
    vtx.append(iB)
    kept.append(new_cell)

    # Mark the midpoint between the two new vertices
    midpoint = 0.5 * (vA + vB)
    events_this_frame.append(("split", midpoint))









def iterate(Vor, pts, vtx, max_step=1000):
    """
    Take one gradient-descent step on all vertices in vtx,
    with adaptive step-size clamping to prevent polygon breakage.
    Performs all phase transitions
    """
    order = Vor.regions.copy()#to ensure cyclic ordering

    #T1
    for pair in short:
        if pair not in Vor.ridge_dict:
            continue #topology changes after each T1, pair might not be valid anymore
        adj = Adjacents(Vor, pair)
        T1(Vor, pair, adj)
        reorder(Vor)
    short.clear()

    #T2
    for s in small:
        T2(Vor, s)
        reorder(Vor)
    small.clear()

    #split
    for ob in oblong:
        split(Vor, ob)
        reorder(Vor)
    oblong.clear()


    eps = 0.02
    min_step = 0

    grads = []
    #compute gradients first
    for i in vtx:
        grad_x, grad_y = energy(Vor, pts, i)
        grads.append([grad_x, grad_y])

    #update vertices
    for vtx_idx, (grad_x, grad_y) in enumerate(grads):

        xstep = grad_x * eps
        ystep = grad_y * eps

        #controlled steps
        Vor.vertices[vtx[vtx_idx], 0] -= np.sign(xstep)*min(max_step, max(min_step, np.abs(xstep)))
        Vor.vertices[vtx[vtx_idx], 1] -= np.sign(ystep)*min(max_step, max(min_step, np.abs(ystep)))



kept = [14]
vtx = find_relevant_vertices(Vor, kept)















#S0=2*np.sqrt(np.pi)


S0=0.8

'''
#find largest degree cell
largest = [len(reg) for reg in Vor.regions]
ind = np.argmax(largest)
iii = list(Vor.point_region).index(ind)
kept = [iii]


vtx = find_relevant_vertices(Vor, kept)

'''




# Store energy history
E_hist = []

# Persistent storage of event markers on energy plot
event_markers = {
    "T1": {"x": [], "y": []},
    "T2": {"x": [], "y": []},
    "split": {"x": [], "y": []}
}


borders = [-1, 2]


# Build a 1x2 layout: tessellation + energy plot
fig, (ax_mesh, ax_E) = plt.subplots(1, 2, figsize=(10, 5))

# Prepare the energy line artist once (faster than replotting)
(line_E,) = ax_E.plot([], [])  # default style; no explicit colors

def init():
    ax_mesh.clear()
    ax_mesh.set_xlim(*borders)
    ax_mesh.set_ylim(*borders)
    ax_mesh.set_aspect("equal")

    ax_E.clear()
    ax_E.set_xlabel("frame")
    ax_E.set_ylabel("Energy")
    ax_E.set_xlim(0, 1)
    ax_E.set_ylim(0, 1)

    global line_E, scat_T1, scat_T2, scat_split

    (line_E,) = ax_E.plot([], [])

    # Create persistent scatter artists
    scat_T1 = ax_E.scatter([], [], marker="o", color = 'red', s=40, zorder=2)
    scat_T2 = ax_E.scatter([], [], marker="o", color = 'green', s=40, zorder=3)
    scat_split = ax_E.scatter([], [], marker="o", color = 'yellow', s=40, zorder=4)

    return (line_E, scat_T1, scat_T2, scat_split)

def update(frame):
    print(f"Progress: {frame+1}/{n_frames} ({100*(frame+1)/n_frames:.1f}%)")
    # (Optional) do multiple physics steps per rendered frame
    # for _ in range(5):
    iterate(Vor, kept, vtx)

    # Compute and store energy
    E = energy(Vor, kept)
    E_hist.append(E)

    # --- redraw tessellation ---
    ax_mesh.clear()
    ax_mesh.set_xlim(*borders)
    ax_mesh.set_ylim(*borders)
    ax_mesh.set_aspect("equal")
    make_nice_tesseltion(Vor, kept, ax=ax_mesh)
    ax_mesh.set_title(f"Energy: {E}")
    ax_E.set_title(f"Cells: {len(kept)}", loc="left")
    plt.suptitle(f"S0={S0}")

    # --- update energy line ---
    x = np.arange(len(E_hist))
    line_E.set_data(x, E_hist)

    # --- record transition events on energy plot ---
    for event_type, _ in events_this_frame:
        event_markers[event_type]["x"].append(len(E_hist) - 1)
        event_markers[event_type]["y"].append(E)

    # Update scatter artists
    if event_markers["T1"]["x"]:
        scat_T1.set_offsets(
            np.column_stack((event_markers["T1"]["x"],
                             event_markers["T1"]["y"]))
        )

    if event_markers["T2"]["x"]:
        scat_T2.set_offsets(
            np.column_stack((event_markers["T2"]["x"],
                             event_markers["T2"]["y"]))
        )

    if event_markers["split"]["x"]:
        scat_split.set_offsets(
            np.column_stack((event_markers["split"]["x"],
                             event_markers["split"]["y"]))
        )

    # Draw event markers
    for ev_type, pos in events_this_frame:
        ax_mesh.scatter(pos[0], pos[1], s=60, marker='o',
                        color={'T1':'red','T2':'green','split':'yellow'}[ev_type],
                        zorder=5)
        ax_mesh.text(pos[0], pos[1], ev_type, color='black',
                     fontsize=8, ha='center', va='center', zorder=6)


    # Clear frame events (important!)
    events_this_frame.clear()

    # Rescale axes neatly
    ax_E.set_xlim(0, max(1, len(E_hist) - 1))
    if np.isfinite(E_hist).all():
        Emin = float(np.min(E_hist))
        Emax = float(np.max(E_hist))
        if Emax == Emin:
            Emax = Emin + 1e-12
        pad = 0.05 * (Emax - Emin)
        ax_E.set_ylim(Emin - pad, Emax + pad)

    return (line_E, scat_T1, scat_T2, scat_split)



# Create animation
n_frames = 200
ani = animation.FuncAnimation(
    fig,
    update,
    frames=n_frames,
    init_func=init,
    blit=False,     # we clear the mesh axis each frame; keep False
    interval=50
)

# Save as GIF (no ffmpeg needed)
ani.save(
    "C:\\Users\\nowak\\OneDrive\\Desktop\\Epithilium frames\\animation104.gif",
    writer=animation.PillowWriter(fps=20),
    dpi=150
)

plt.close(fig)
print("Saved!")





Progress: 1/200 (0.5%)
Progress: 2/200 (1.0%)
Progress: 3/200 (1.5%)
Progress: 4/200 (2.0%)
Progress: 5/200 (2.5%)
Progress: 6/200 (3.0%)
Progress: 7/200 (3.5%)
Progress: 8/200 (4.0%)
Progress: 9/200 (4.5%)
Progress: 10/200 (5.0%)
Progress: 11/200 (5.5%)
Progress: 12/200 (6.0%)
Progress: 13/200 (6.5%)
Progress: 14/200 (7.0%)
Progress: 15/200 (7.5%)
Progress: 16/200 (8.0%)
Progress: 17/200 (8.5%)
Progress: 18/200 (9.0%)
Progress: 19/200 (9.5%)
Progress: 20/200 (10.0%)
Progress: 21/200 (10.5%)
Progress: 22/200 (11.0%)
Progress: 23/200 (11.5%)
Progress: 24/200 (12.0%)
Progress: 25/200 (12.5%)
Progress: 26/200 (13.0%)
Progress: 27/200 (13.5%)
Progress: 28/200 (14.0%)
Progress: 29/200 (14.5%)
Progress: 30/200 (15.0%)
Progress: 31/200 (15.5%)
Progress: 32/200 (16.0%)
Progress: 33/200 (16.5%)
Progress: 34/200 (17.0%)
Progress: 35/200 (17.5%)
Progress: 36/200 (18.0%)
Progress: 37/200 (18.5%)
Progress: 38/200 (19.0%)
Progress: 39/200 (19.5%)
Progress: 40/200 (20.0%)
Progress: 41/200 (20.5%)
Pro